In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data")

# Load datasets
characters = pd.read_csv(DATA_DIR / "Characters.csv", delimiter=';')
hp1 = pd.read_csv(DATA_DIR / "Harry Potter 1.csv", delimiter=';')
hp2 = pd.read_csv(DATA_DIR / "Harry Potter 2.csv", delimiter=';')
hp3 = pd.read_csv(DATA_DIR / "Harry Potter 3.csv", delimiter=';')
spells = pd.read_csv(DATA_DIR / "Spells.csv", delimiter=';')

#Show shapes of datasets
print('Characters shape:', characters.shape)
print('First batch of sentences shape:',hp1.shape)
print('Second batch of sentences shape:',hp2.shape)
print('Third batch of sentences shape:',hp3.shape)
print('Spells shape:',spells.shape)


In [ ]:
# Info and null values of characters dataset
characters.info()
characters.isnull().sum()

In [ ]:
# Distribution per House

characters["House"].value_counts().plot(kind="bar")
plt.title("Distribution per House")
plt.xlabel("House")
plt.ylabel("Count")
plt.show()

In [ ]:
# Distribution per Gender

characters["Gender"].value_counts().plot(kind="bar")
plt.title("Distribution per Gender")
plt.xlabel("Gender")
plt.ylabel("Count")
plt.show()

In [ ]:
#Merging sentences datasets for further evaluation

# Rename HP3's columns
hp3 = hp3.rename(columns={"CHARACTER": "Character", "SENTENCE": "Sentence"})

# Eliminate blank spaces and make uppercase
hp1['Character'] = hp1['Character'].str.upper().str.strip()
hp2['Character'] = hp2['Character'].str.upper().str.strip()
hp3['Character'] = hp3['Character'].str.upper().str.strip()

hp2 = hp2[['Character', 'Sentence']]
hp3 = hp3[['Character', 'Sentence']]

# Concat everything
hp_all = pd.concat([hp1, hp2, hp3], ignore_index=True)

# Replace 'RON' with 'RONALD' in Character column
hp_all.loc[hp_all['Character'] == 'RON', 'Character'] = 'RONALD'
hp_all.loc[hp_all['Character'] == 'MRS. WEASLEY', 'Character'] = 'MOLLY WEASLEY'

print('All sentences shape:', hp_all.shape)

In [ ]:
# Count lines per character (without repeating character)
lines_per_character = hp_all['Character'].value_counts()

# Graph
plt.figure(figsize=(12, 6))
lines_per_character.plot(kind='bar')
plt.title('Amount of lines per character')
plt.xlabel('Character')
plt.ylabel('Number of lines')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution per Spell

spells["Type"].value_counts().plot(kind="bar")
plt.title("Distribution per Spell")
plt.xlabel("Spell Type")
plt.ylabel("Count")
plt.show()

In [ ]:
# Amount of times that each spell is said in the sentences dataset

spell_counts = []
for incantation in spells['Incantation'].dropna().unique():
    # Ignore empty incantations
    if incantation.strip() == '' or incantation.lower() == 'unknown':
        continue
    count = hp_all['Sentence'].str.contains(incantation, case=False, na=False).sum()
    spell_counts.append({'Incantation': incantation, 'TimesSaid': count})

# Create df
spells_mentions_df = pd.DataFrame(spell_counts)

print(spells_mentions_df.head(20))

In [ ]:
# Filter spells that are said at least once
spells_said = spells_mentions_df[spells_mentions_df['TimesSaid'] > 0]

print(spells_said.head)

In [ ]:
# Extract all factions from the Loyalty column
faction_counts = {}
for loyalties in characters['Loyalty'].dropna():
    # Separate by ‘|’ and remove spaces
    factions = [f.strip() for f in loyalties.split('|') if f.strip()]
    for faction in factions:
        faction_counts[faction] = faction_counts.get(faction, 0) + 1

# Convert to DataFrame for graphing
faction_df = pd.DataFrame(list(faction_counts.items()), columns=['Faction', 'NumCharacters'])
faction_df = faction_df.sort_values('NumCharacters', ascending=False)

# graphing
plt.figure(figsize=(12, 6))
plt.bar(faction_df['Faction'], faction_df['NumCharacters'])
plt.xticks(rotation=90)
plt.title('Cantidad de personajes leales por facción')
plt.xlabel('Facción')
plt.ylabel('Número de personajes leales')
plt.tight_layout()
plt.show()

In [ ]:
# Create a dataset with Character and Incantation if the Sentence contains an Incantation from spells_said (regardless of uppercase/lowercase letters)
rows = []
for _, row in hp_all.iterrows():
    for incantation in spells_said['Incantation']:
        if pd.notna(row['Sentence']) and pd.notna(incantation):
            if incantation.lower() in row['Sentence'].lower():
                rows.append({'Character': row['Character'], 'Incantation': incantation})
dataset_character_incantation = pd.DataFrame(rows)

print(dataset_character_incantation.head)

In [ ]:
# Generate ‘Incantations’ column in characters
incantations_list = []
for _, char_row in characters.iterrows():
    name = str(char_row['Name'])
    # Search for all characters in dataset_character_incantation that are in Name (ignoring upper/lower case)
    incantations = dataset_character_incantation[
        dataset_character_incantation['Character'].apply(lambda c: c.lower() in name.lower())
    ]['Incantation'].unique()
    # Join the incantations with ‘ | ’ as in Loyalty, or leave blank if there are none
    incantations_str = ' | '.join(incantations) if len(incantations) > 0 else ''
    incantations_list.append(incantations_str)

# Add the column to the original DataFrame
characters['Incantations'] = incantations_list

print(characters[['Name', 'Incantations']].head(5))